<a href="https://colab.research.google.com/github/Pedroleaoo/Mercado_Games/blob/main/dashboard_steam_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dashboard — Mercado de Games na Steam
**Projeto Final · Análise de Dados e BI** — Bruno Sena, Matheus Gabriel e Pedro Vitor

Este notebook lê as bases limpas (`games_clean.csv` e `language_clean.csv`), calcula as 6 métricas do projeto e monta uma apresentação interativa (Plotly) com a resposta de cada pergunta de negócio e um slide final com a solução para a publisher.

**Como usar:** execute as células em ordem (Ambiente de execução → Executar tudo). No fim, a apresentação aparece na tela e também é salva como `dashboard_steam.html`. Para apresentar, abra o HTML no navegador, tecle F11 e navegue com as setas ← →.

## 1. Bibliotecas

In [13]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import spearmanr
from IPython.display import HTML, display

pd.options.display.float_format = '{:,.2f}'.format
print('Bibliotecas carregadas')

Bibliotecas carregadas


## 2. Carregar os dados


In [14]:
from google.colab import drive
drive.mount('/content/drive')

PASTA = '/content/drive/MyDrive/projeto_steam/'

g = pd.read_csv(PASTA + 'games_clean.csv')
lang = pd.read_csv(PASTA + 'language_clean.csv')

print(f'games_clean : {g.shape[0]:,} jogos × {g.shape[1]} colunas')
print(f'language_clean: {lang.shape[0]} idiomas')
g[['Name', 'release_year', 'estimated_owners_mid', 'is_hit', 'price_band', 'genres_str', 'pct_positive']].head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
games_clean : 125,854 jogos × 41 colunas
language_clean: 29 idiomas


,Name,release_year,estimated_owners_mid,is_hit,price_band,genres_str,pct_positive
0,Supipara - Chapter 1 Spring Has Come!,2016,"10,000.00",False,até US$10,Adventure,0.99
1,Mystery Solitaire The Black Raven,2019,"10,000.00",False,até US$10,Casual,NaN
2,버튜버 파라노이아 - Vtuber Paranoia,2024,"10,000.00",False,até US$10,"Casual,Indie,Simulation",NaN
3,Maze Quest VR,2025,"10,000.00",False,até US$10,Action,NaN
4,Agony VR,2023,"10,000.00",False,US$10–30,"Action,Adventure",NaN


## 3. Configurações, paleta e funções auxiliares

In [15]:
MIN_JOGOS   = 30
MIN_AVAL    = 50
P1, P2      = '2020–2022', '2023–2025'
MIN_TAG     = 30
TAXA_BASE   = g['is_hit'].mean() * 100

MERCADOS = {
    'Inglês': 'lang_english',
    'Alemão': 'lang_german',
    'Japonês': 'lang_japanese',
    'Chinês simplificado': 'lang_simplified_chinese',
}
NOME_SURVEY = {'Inglês': 'English', 'Alemão': 'German',
               'Japonês': 'Japanese', 'Chinês simplificado': 'Simplified Chinese'}

SURVEY_OUT26 = {'Inglês': 39.04, 'Chinês simplificado': 21.98, 'Alemão': 2.89, 'Japonês': 2.47}

ORDEM_PRECO = ['gratuito', 'até US$10', 'US$10–30', 'acima US$30']

AZUL, LARANJA, VERDE, VERMELHO, CINZA, ROXO = '#2f5d7c', '#c8553d', '#4d7c6f', '#a23b3b', '#b9b5ab', '#7a6a8a'
INK, MUTED, LINHA = '#1f2428', '#6b6f73', '#e4e0d6'
SEQ = [AZUL, LARANJA, VERDE, '#d6a84a', ROXO, CINZA]
ESCALA = [[0, '#f4f1ea'], [1, AZUL]]

def br(txt):
    return txt.replace(',', '§').replace('.', ',').replace('§', '.')
def n0(x):  return br(f'{x:,.0f}')
def n1(x):  return br(f'{x:,.1f}')
def n2(x):  return br(f'{x:,.2f}')
def pct(x, d=1): return br(f'{x:,.{d}f}') + '%'
def pp(x, d=2): return br(f'{x:+,.{d}f}') + ' p.p.'

def estilo(fig, titulo, altura=430, legenda=True):
    fig.update_layout(
        meta=titulo, template='plotly_white', height=altura, colorway=SEQ,
        paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)',
        font=dict(family='Inter, Segoe UI, Arial, sans-serif', size=12, color=INK),
        margin=dict(l=50, r=20, t=44 if legenda else 16, b=44), showlegend=legenda,
        legend=dict(orientation='h', y=1.1, x=0, font=dict(size=12)),
        hoverlabel=dict(bgcolor='white', font_size=12))
    fig.update_xaxes(gridcolor=LINHA, zeroline=False, linecolor=LINHA)
    fig.update_yaxes(gridcolor=LINHA, zeroline=False, linecolor=LINHA)
    fig.update_annotations(font_size=12, font_color=MUTED)
    return fig

def e_lista(xs):
    xs = [str(x) for x in xs]
    return xs[0] if len(xs) == 1 else ', '.join(xs[:-1]) + ' e ' + xs[-1]

def forca(r):
    r = abs(r)
    return 'inexistente ou muito fraca' if r < 0.1 else 'fraca' if r < 0.3 else 'moderada' if r < 0.5 else 'forte'

def explodir(df, col, extra=()):
    cols = ['AppID', 'estimated_owners_mid', 'is_hit'] + [c for c in extra if c not in ('AppID', 'estimated_owners_mid', 'is_hit')]
    d = df.loc[df[col].notna(), cols + [col]].copy()
    d['item'] = d[col].str.split(',')
    d = d.explode('item')
    d['item'] = d['item'].str.strip()
    return d.drop(columns=col)

def resumo_grupo(d, chave):
    """Métricas 1 e 2 por grupo: jogos, média, mediana, mediana >20 mil e taxa de hits."""
    def _med20(s):
        s = s[s > 20000]
        return s.median() if len(s) else np.nan
    return (d.groupby(chave)
              .agg(jogos=('AppID', 'nunique'),
                   donos=('estimated_owners_mid', 'sum'),
                   media=('estimated_owners_mid', 'mean'),
                   mediana=('estimated_owners_mid', 'median'),
                   mediana_20k=('estimated_owners_mid', _med20),
                   taxa_hits=('is_hit', lambda s: s.mean() * 100))
              .reset_index())

def tabela_html(df):
    return df.to_html(index=False, border=0, classes='tabela', escape=False)

print(f'Taxa geral de hits da base: {TAXA_BASE:.2f}%  |  jogos: {len(g):,}')

Taxa geral de hits da base: 7.01%  |  jogos: 125,854


## 4. Pergunta 1 — Gênero por mercado de idioma (métricas 1, 3 e 6)

In [4]:
def pergunta1():
    partes = []
    for mercado, flag in MERCADOS.items():
        sub = g[g[flag]]
        total_donos = sub.loc[sub['genres_str'].notna(), 'estimated_owners_mid'].sum()
        d = explodir(sub, 'genres_str')
        r = resumo_grupo(d, 'item').rename(columns={'item': 'genero'})
        r['participacao'] = r['donos'] / total_donos * 100
        r['mercado'] = mercado
        partes.append(r)
    base = pd.concat(partes, ignore_index=True)

    top3 = (base.sort_values(['mercado', 'participacao'], ascending=[True, False]).groupby('mercado').head(3).copy())
    top3['rank'] = top3.groupby('mercado').cumcount() + 1
    top3['mercado'] = pd.Categorical(top3['mercado'], list(MERCADOS), ordered=True)
    top3 = top3.sort_values(['mercado', 'rank'])

    piv = base.pivot(index='genero', columns='mercado', values='participacao')[list(MERCADOS)]
    piv = piv.loc[piv.mean(axis=1).sort_values(ascending=True).index]
    f1 = go.Figure(go.Heatmap(z=piv.values, x=piv.columns, y=piv.index, colorscale=ESCALA, showscale=False,
                              text=np.round(piv.values, 1), texttemplate='%{text}%',
                              hovertemplate='%{y} · %{x}: %{z:.1f}%<extra></extra>', xgap=2, ygap=2))
    estilo(f1, 'Participação de cada gênero nos donos estimados de cada mercado (% dos donos)', 470, False)

    tot = len(g)
    oferta = {m: g[f].sum() / tot * 100 for m, f in MERCADOS.items()}
    usuarios = {m: float(lang.loc[lang['Language_padronizado'] == NOME_SURVEY[m], 'PercApr22'].iloc[0]) for m in MERCADOS}
    f2 = go.Figure()
    f2.add_bar(name='% dos jogos com suporte ao idioma', x=list(MERCADOS), y=list(oferta.values()), marker_color=AZUL,
               text=[pct(v) for v in oferta.values()], textposition='outside')
    f2.add_bar(name='% dos usuários da Steam no idioma (abr/2022)', x=list(MERCADOS), y=list(usuarios.values()), marker_color=LARANJA,
               text=[pct(v) for v in usuarios.values()], textposition='outside')
    f2.update_layout(barmode='group', yaxis_range=[0, 105], yaxis_title='%')
    estilo(f2, 'Oferta de jogos x peso do idioma entre os usuários', 380)

    tab = top3[['mercado', 'rank', 'genero', 'participacao', 'mediana', 'mediana_20k', 'taxa_hits', 'jogos']].copy()
    tab['participacao'] = tab['participacao'].map(pct); tab['mediana'] = tab['mediana'].map(n0)
    tab['mediana_20k'] = tab['mediana_20k'].map(n0); tab['taxa_hits'] = tab['taxa_hits'].map(pct); tab['jogos'] = tab['jogos'].map(n0)
    tab.columns = ['Mercado', '#', 'Gênero', 'Participação (% donos)', 'Mediana donos', 'Mediana donos (>20 mil)', 'Taxa de hits', 'Jogos']

    lideres = {m: top3[top3['mercado'] == m] for m in MERCADOS}
    conjuntos = [frozenset(t['genero']) for t in lideres.values()]
    ing = lideres['Inglês']
    if len(set(conjuntos)) == 1:
        headline = f"{e_lista(ing['genero'])} lideram nos quatro mercados; o que muda de um mercado para outro é o tamanho do idioma, não o gênero."
    else:
        headline = 'Os gêneros líderes mudam de um mercado de idioma para outro (veja o mapa de calor).'
    mais_prox = min(MERCADOS, key=lambda m: abs(oferta[m] - usuarios[m]))
    stats = [(pct(ing.iloc[0]['participacao']), f"dos donos do mercado em inglês estão em jogos de {ing.iloc[0]['genero']}"),
             (pct(oferta['Inglês']), f"dos jogos têm inglês, contra {pct(usuarios['Inglês'])} dos usuários"),
             (pct(oferta['Chinês simplificado']), f"dos jogos têm chinês simplificado, contra {pct(usuarios['Chinês simplificado'])} dos usuários")]
    leitura = [f"<b>{m}</b>: " + ' · '.join(f"{r.genero} {pct(r.participacao)}" for r in t.itertuples()) for m, t in lideres.items()]
    leitura.append(f"Só o <b>{mais_prox}</b> tem oferta de jogos próxima do peso do idioma entre os usuários; nos outros, a oferta é bem maior.")
    return dict(id='p1', curto='Mercado', titulo='Gênero por mercado',
                pergunta='Quais gêneros têm maior público e participação em cada mercado de idioma, e isso condiz com o tamanho do idioma na Steam?',
                headline=headline, stats=stats, leitura=leitura, figs=[f1, f2], tabelas=[tab],
                nota='Idioma suportado é aproximação de mercado (a base não tem vendas por região). Participação = donos do gênero ÷ donos dos jogos com gênero naquele idioma; um jogo conta em cada gênero, então a soma passa de 100%. Os dois percentuais do gráfico de oferta têm bases diferentes: a comparação é qualitativa.',
                dados=dict(oferta=oferta, usuarios=usuarios, lideres=lideres))

P1_ = pergunta1()
print('Pergunta 1 ok')

Pergunta 1 ok


## 5. Pergunta 2 — Nota x público (métrica 4)

In [5]:
def pergunta2():
    a = g[g['total_reviews'] >= MIN_AVAL].copy()
    rho, p = spearmanr(a['pct_positive'], a['estimated_owners_mid'])
    a['faixa'] = pd.cut(a['pct_positive'] * 100, [0, 50, 70, 80, 90, 100.01],
                        labels=['<50%', '50–70%', '70–80%', '80–90%', '90–100%'], right=False)
    fa = a.groupby('faixa', observed=True).agg(jogos=('AppID', 'count'), media=('estimated_owners_mid', 'mean'),
            mediana=('estimated_owners_mid', 'median'), taxa_hits=('is_hit', lambda s: s.mean() * 100)).reset_index()

    f1 = make_subplots(specs=[[{'secondary_y': True}]])
    f1.add_bar(x=fa['faixa'].astype(str), y=fa['media'], name='Público médio (donos)', marker_color=AZUL,
               text=[n0(v) for v in fa['media']], textposition='outside')
    f1.add_scatter(x=fa['faixa'].astype(str), y=fa['taxa_hits'], name='Taxa de hits (%)', mode='lines+markers',
                   line=dict(color=LARANJA, width=3), secondary_y=True)
    f1.update_yaxes(title_text='Donos estimados (média)', secondary_y=False)
    f1.update_yaxes(title_text='Taxa de hits (%)', secondary_y=True, showgrid=False)
    f1.update_xaxes(title_text='% de avaliações positivas na Steam')
    estilo(f1, 'Público médio e taxa de hits por faixa de % de avaliações positivas')

    s_ = a.sample(min(6000, len(a)), random_state=1)
    f2 = go.Figure(go.Scattergl(x=s_['pct_positive'] * 100, y=s_['estimated_owners_mid'].clip(lower=1), mode='markers',
                                marker=dict(size=5, color=AZUL, opacity=0.3), hovertext=s_['Name']))
    f2.update_yaxes(type='log', title='Donos estimados (escala log)'); f2.update_xaxes(title='% de avaliações positivas')
    estilo(f2, 'Cada ponto é um jogo (amostra de 6 mil)', 380, False)

    m = g[g['Metacritic score'] > 0].copy()
    rho_m, _ = spearmanr(m['Metacritic score'], m['estimated_owners_mid'])
    m['faixa'] = pd.cut(m['Metacritic score'], [0, 60, 80, 101], labels=['<60', '60–79', '80+'], right=False)
    fm = m.groupby('faixa', observed=True).agg(jogos=('AppID', 'count'), media=('estimated_owners_mid', 'mean'),
            mediana=('estimated_owners_mid', 'median'), taxa_hits=('is_hit', lambda s: s.mean() * 100)).reset_index()
    f3 = go.Figure(go.Bar(x=fm['faixa'].astype(str), y=fm['media'], marker_color=[CINZA, CINZA, LARANJA],
                          text=[n0(v) for v in fm['media']], textposition='outside'))
    f3.update_yaxes(title='Donos estimados (média)'); f3.update_xaxes(title='Nota no Metacritic')
    estilo(f3, f'Público médio por faixa de Metacritic ({n0(len(m))} jogos com nota)', 380, False)

    r_med = fm.loc[fm['faixa'] == '80+', 'media'].iloc[0] / fm.loc[fm['faixa'] == '<60', 'media'].iloc[0]
    r_mdn = fm.loc[fm['faixa'] == '80+', 'mediana'].iloc[0] / fm.loc[fm['faixa'] == '<60', 'mediana'].iloc[0]

    def fmt(df):
        t = df.copy(); t['faixa'] = t['faixa'].astype(str)
        for c in ['jogos', 'media', 'mediana']: t[c] = t[c].map(n0)
        t['taxa_hits'] = t['taxa_hits'].map(pct)
        return t
    t1 = fmt(fa); t1.columns = ['% positivas', 'Jogos', 'Público médio', 'Mediana', 'Taxa de hits']
    t2 = fmt(fm); t2.columns = ['Metacritic', 'Jogos', 'Público médio', 'Mediana', 'Taxa de hits']

    headline = (f"A % de avaliações positivas tem relação {forca(rho)} com o público (ρ = {n2(rho)}); "
                f"a nota do Metacritic tem relação {forca(rho_m)} (ρ = {n2(rho_m)}).")
    stats = [(f'ρ = {n2(rho)}', f'Steam: % positivas x donos ({n0(len(a))} jogos com ≥ {MIN_AVAL} avaliações)'),
             (f'ρ = {n2(rho_m)}', f'Metacritic x donos ({n0(len(m))} jogos com nota)'),
             (f'{n1(r_med)}×', 'público médio dos jogos com nota 80+ em relação aos de nota abaixo de 60')]
    leitura = [f"Correlação de Spearman: quanto mais perto de 0, menos os dois números andam juntos. Na Steam, ρ = {n2(rho)}.",
               f"No Metacritic, jogos 80+ têm {n1(r_med)}× o público médio dos jogos &lt; 60 (mediana: {n1(r_mdn)}×): a meta de pelo menos 2× {'foi atingida' if r_med >= 2 else 'não foi atingida'}.",
               "É associação, não causa: jogos com nota no Metacritic costumam ser de estúdios maiores."]
    return dict(id='p2', curto='Nota', titulo='Nota x público', pergunta='Os jogos mais bem avaliados têm mais público?',
                headline=headline, stats=stats, leitura=leitura, figs=[f1, f2, f3], tabelas=[t1, t2],
                nota=f'Medida principal: % de avaliações positivas na Steam (jogos com ≥ {MIN_AVAL} avaliações). O Metacritic é complemento e só vale para os jogos com nota maior que zero.',
                dados=dict(rho=rho, rho_m=rho_m, r_med=r_med))

P2_ = pergunta2()
print('Pergunta 2 ok')

Pergunta 2 ok


## 6. Pergunta 3 — Gênero x faixa de preço (métrica 2)

In [6]:
def pergunta3():
    d = explodir(g, 'genres_str', extra=['price_band'])
    r = (d.groupby(['item', 'price_band']).agg(jogos=('AppID', 'nunique'), hits=('is_hit', 'sum'),
                                               mediana=('estimated_owners_mid', 'median')).reset_index())
    r['taxa_hits'] = r['hits'] / r['jogos'] * 100
    r = r[r['jogos'] >= MIN_JOGOS].copy()
    top5 = r.sort_values('taxa_hits', ascending=False).head(5).copy()

    rot = [f'{a} · {b}' for a, b in zip(top5['item'], top5['price_band'])][::-1]
    f1 = go.Figure(go.Bar(y=rot, x=top5['taxa_hits'][::-1], orientation='h', marker_color=[CINZA] * 4 + [LARANJA],
                          text=[pct(v) for v in top5['taxa_hits'][::-1]], textposition='outside'))
    f1.add_vline(x=TAXA_BASE, line_dash='dash', line_color=INK, annotation_text=f'Base: {pct(TAXA_BASE)}', annotation_position='top')
    f1.update_xaxes(title='Taxa de hits (%)', range=[0, top5['taxa_hits'].max() * 1.2])
    estilo(f1, f'As 5 combinações com maior taxa de hits (mínimo de {MIN_JOGOS} jogos)', 380, False)

    piv = r.pivot(index='item', columns='price_band', values='taxa_hits').reindex(columns=ORDEM_PRECO)
    piv = piv.loc[piv.max(axis=1).sort_values().index]
    f2 = go.Figure(go.Heatmap(z=piv.values, x=piv.columns, y=piv.index, colorscale=ESCALA, showscale=False,
                              text=np.round(piv.values, 1), texttemplate='%{text}%',
                              hovertemplate='%{y} · %{x}: %{z:.1f}%<extra></extra>', xgap=2, ygap=2))
    estilo(f2, 'Taxa de hits (%) por gênero e faixa de preço; células vazias têm menos de 30 jogos', 440, False)

    t = top5.copy()
    t['taxa_hits'] = t['taxa_hits'].map(pct)
    for c in ['jogos', 'hits', 'mediana']: t[c] = t[c].map(n0)
    t.columns = ['Gênero', 'Faixa de preço', 'Jogos', 'Hits', 'Mediana donos', 'Taxa de hits']
    top = top5.iloc[0]
    headline = f"{top['item']} · {top['price_band']} é a combinação com maior chance de virar hit: {pct(top['taxa_hits'])}, {n1(top['taxa_hits'] / TAXA_BASE)}× a taxa geral da base."
    stats = [(pct(top['taxa_hits']), f"taxa de hits de {top['item']} · {top['price_band']}"),
             (f"{n1(top['taxa_hits'] / TAXA_BASE)}×", f'a taxa geral da base ({pct(TAXA_BASE)})'),
             (n0(top['jogos']), 'jogos nessa combinação')]
    leitura = [f"{x.item} · {x.price_band}: <b>{pct(x.taxa_hits)}</b> ({n0(x.jogos)} jogos)" for x in top5.itertuples()]
    leitura.append('Hit = mais de 100 mil donos estimados. Entram só combinações com pelo menos 30 jogos.')
    return dict(id='p3', curto='Preço', titulo='Gênero x preço',
                pergunta='Qual combinação de gênero e faixa de preço costuma virar grande sucesso de público?',
                headline=headline, stats=stats, leitura=leitura, figs=[f1, f2], tabelas=[t],
                nota='A linha tracejada é a taxa geral de hits da base. Um jogo conta em cada um de seus gêneros.',
                dados=dict(top5=top5, todas=r))

P3_ = pergunta3()
print('Pergunta 3 ok')

Pergunta 3 ok


## 7. Pergunta 4 — Multijogador x um jogador (métricas 1 e 2)

In [7]:
def pergunta4():
    sub = g[g['tags_str'].notna()].copy()
    listas = sub['tags_str'].str.split(',').map(lambda L: [x.strip() for x in L])
    linhas = []
    for nome in ['Singleplayer', 'Multiplayer', 'Co-op']:
        s_ = sub[listas.map(lambda L: nome in L)]
        if len(s_) >= MIN_JOGOS:
            x = s_['estimated_owners_mid']
            linhas.append(dict(grupo=nome, jogos=len(s_), media=x.mean(), mediana=x.median(),
                               mediana_20k=x[x > 20000].median(), taxa_hits=s_['is_hit'].mean() * 100))
    r = pd.DataFrame(linhas)
    cor = {'Singleplayer': CINZA, 'Multiplayer': LARANJA, 'Co-op': AZUL}

    f1 = go.Figure(go.Bar(x=r['grupo'], y=r['taxa_hits'], marker_color=[cor[x] for x in r['grupo']],
                          text=[pct(v) for v in r['taxa_hits']], textposition='outside'))
    f1.add_hline(y=TAXA_BASE, line_dash='dash', line_color=INK, annotation_text=f'Base: {pct(TAXA_BASE)}')
    f1.update_yaxes(title='Taxa de hits (%)', range=[0, r['taxa_hits'].max() * 1.25])
    estilo(f1, 'Taxa de hits por modo de jogo', 420, False)

    f2 = make_subplots(rows=1, cols=2, subplot_titles=('Mediana de donos (todos os jogos)', 'Mediana de donos (jogos com mais de 20 mil)'))
    f2.add_bar(x=r['grupo'], y=r['mediana'], marker_color=CINZA, text=[n0(v) for v in r['mediana']], textposition='outside', row=1, col=1)
    f2.add_bar(x=r['grupo'], y=r['mediana_20k'], marker_color=AZUL, text=[n0(v) for v in r['mediana_20k']], textposition='outside', row=1, col=2)
    estilo(f2, 'Público típico por modo de jogo', 380, False)

    t = r.copy()
    for c in ['jogos', 'media', 'mediana', 'mediana_20k']: t[c] = t[c].map(n0)
    t['taxa_hits'] = t['taxa_hits'].map(pct)
    t.columns = ['Grupo', 'Jogos', 'Público médio', 'Mediana', 'Mediana (>20 mil)', 'Taxa de hits']

    rr = r.sort_values('taxa_hits', ascending=False)
    top = rr.iloc[0]
    sp = r.loc[r['grupo'] == 'Singleplayer', 'taxa_hits']
    if top['grupo'] != 'Singleplayer' and len(sp):
        headline = f"{top['grupo']} tem a maior chance de hit ({pct(top['taxa_hits'])}), {n1(top['taxa_hits'] / sp.iloc[0])}× a dos jogos só de um jogador ({pct(sp.iloc[0])})."
    else:
        headline = f"{top['grupo']} tem a maior taxa de hits ({pct(top['taxa_hits'])}), contra {pct(TAXA_BASE)} da base."
    stats = [(pct(x.taxa_hits), f'{x.grupo}: taxa de hits ({n0(x.jogos)} jogos)') for x in r.itertuples()]
    acima = r[r['taxa_hits'] > TAXA_BASE]['grupo'].tolist()
    leitura = [f"Taxa geral da base: {pct(TAXA_BASE)}. Grupos acima dela: <b>{e_lista(acima) if acima else 'nenhum'}</b>.",
               'Mediana de donos: ' + ' · '.join(f'{x.grupo} {n0(x.mediana)}' for x in r.itertuples()) + '.',
               'Um jogo pode ter mais de uma tag, então os grupos se sobrepõem. Só entram jogos com tags (cerca de 66% da base).']
    return dict(id='p4', curto='Multiplayer', titulo='Multijogador x um jogador',
                pergunta='Jogos com modo multijogador ou cooperativo têm mais público e mais chance de virar hit do que jogos só de um jogador?',
                headline=headline, stats=stats, leitura=leitura, figs=[f1, f2], tabelas=[t],
                nota='A idade declarada cobre só 1% dos jogos e não foi usada. Como a mediana de donos fica na faixa mais baixa para a maioria dos jogos, a taxa de hits é a medida mais informativa aqui.',
                dados=dict(r=r))

P4_ = pergunta4()
print('Pergunta 4 ok')

Pergunta 4 ok


## 8. Pergunta 5 — Tendências 2020–2022 x 2023–2025 e Survey 2022 x 2026 (métricas 5 e 6)

In [8]:
def variacao(col, minimo):
    sub = g[g['release_year'].between(2020, 2025)].copy()
    sub['periodo'] = np.where(sub['release_year'] <= 2022, P1, P2)
    base = sub[sub[col].notna()].groupby('periodo').size()
    d = explodir(sub, col, extra=['periodo'])
    c = d.groupby(['item', 'periodo']).agg(jogos=('AppID', 'nunique'), mediana=('estimated_owners_mid', 'median')).reset_index()
    c['part'] = c['jogos'] / c['periodo'].map(base) * 100
    p = c.pivot(index='item', columns='periodo', values=['jogos', 'part', 'mediana']).dropna()
    out = pd.DataFrame({'jogos_p1': p[('jogos', P1)], 'jogos_p2': p[('jogos', P2)],
                        'part_p1': p[('part', P1)], 'part_p2': p[('part', P2)],
                        'med_p1': p[('mediana', P1)], 'med_p2': p[('mediana', P2)]})
    out['delta'] = out['part_p2'] - out['part_p1']
    out = out[(out['jogos_p1'] >= minimo) & (out['jogos_p2'] >= minimo)]
    return out.reset_index().rename(columns={'item': 'nome'}).sort_values('delta', ascending=False)

def barras_delta(df, altura):
    df = df.sort_values('delta')
    f = go.Figure(go.Bar(y=df['nome'], x=df['delta'], orientation='h',
                         marker_color=[VERDE if v >= 0 else VERMELHO for v in df['delta']],
                         text=[pp(v) for v in df['delta']], textposition='outside'))
    f.add_vline(x=0, line_color=INK)
    lim = df['delta'].abs().max() * 1.35
    f.update_xaxes(title='Variação da participação nos lançamentos (pontos percentuais)', range=[-lim, lim])
    return f

def pergunta5():
    gen = variacao('genres_str', 1)
    tag = variacao('tags_str', MIN_TAG)
    tag_sel = pd.concat([tag.head(3), tag.tail(3)]).drop_duplicates('nome')
    gen_sel = pd.concat([gen.head(3), gen.tail(3)]).drop_duplicates('nome')

    f1 = estilo(barras_delta(gen, 450), f'Gêneros: variação da participação nos lançamentos, {P1} → {P2}', 450, False)
    f2 = estilo(barras_delta(tag_sel, 380), 'As 3 tags que mais ganharam e as 3 que mais perderam participação', 380, False)

    surv = pd.DataFrame({'idioma': list(MERCADOS),
                         'abr22': [float(lang.loc[lang['Language_padronizado'] == NOME_SURVEY[m], 'PercApr22'].iloc[0]) for m in MERCADOS],
                         'out26': [SURVEY_OUT26[m] for m in MERCADOS]})
    surv['var'] = surv['out26'] - surv['abr22']
    f3 = go.Figure()
    f3.add_bar(name='Abril/2022', x=surv['idioma'], y=surv['abr22'], marker_color=CINZA, text=[pct(v) for v in surv['abr22']], textposition='outside')
    f3.add_bar(name='Outubro/2026', x=surv['idioma'], y=surv['out26'], marker_color=AZUL, text=[pct(v) for v in surv['out26']], textposition='outside')
    for _, r in surv.iterrows():
        f3.add_annotation(x=r['idioma'], y=max(r['abr22'], r['out26']) + 6, text=f"<b>{pp(r['var'])}</b>", showarrow=False,
                          font=dict(color=VERDE if r['var'] >= 0 else VERMELHO, size=12))
    f3.update_layout(barmode='group', yaxis_range=[0, 52], yaxis_title='% dos usuários da Steam')
    estilo(f3, 'Steam Survey: peso de cada idioma entre os usuários')

    def tabela(df):
        t = df.copy()
        t['jogos'] = t['jogos_p1'].map(n0) + ' → ' + t['jogos_p2'].map(n0)
        t['part'] = t['part_p1'].map(lambda v: pct(v, 2)) + ' → ' + t['part_p2'].map(lambda v: pct(v, 2))
        t['mediana'] = t['med_p1'].map(n0) + ' → ' + t['med_p2'].map(n0)
        t['delta'] = t['delta'].map(pp)
        return t[['nome', 'jogos', 'part', 'delta', 'mediana']].rename(columns={
            'nome': 'Gênero/Tag', 'jogos': 'Lançamentos', 'part': 'Participação', 'delta': 'Variação', 'mediana': 'Mediana de donos'})
    ts = tabela(pd.concat([gen_sel, tag_sel]))
    ts.insert(0, 'Tipo', ['Gênero'] * len(gen_sel) + ['Tag'] * len(tag_sel))

    ganha = gen.head(3); perde = gen.tail(3).iloc[::-1]
    mudou = surv[surv['var'].abs() > 2]
    maior = surv.loc[surv['var'].abs().idxmax()]
    headline = f"{e_lista(ganha['nome'])} ganham espaço nos lançamentos; {e_lista(perde['nome'])} perdem."
    stats = [(pp(ganha.iloc[0]['delta']), f"{ganha.iloc[0]['nome']}: maior ganho de participação entre os gêneros"),
             (pp(perde.iloc[0]['delta']), f"{perde.iloc[0]['nome']}: maior perda de participação entre os gêneros"),
             (pp(maior['var']), f"{maior['idioma']} entre os usuários da Steam, abr/2022 → out/2026")]
    leitura = [f"<b>Gêneros em alta:</b> {', '.join(f'{x.nome} ({pp(x.delta)})' for x in ganha.itertuples())}.",
               f"<b>Gêneros em queda:</b> {', '.join(f'{x.nome} ({pp(x.delta)})' for x in perde.itertuples())}.",
               f"<b>Tags em alta:</b> {', '.join(f'{x.nome} ({pp(x.delta)})' for x in tag.head(3).itertuples())}.",
               f"<b>Tags em queda:</b> {', '.join(f'{x.nome} ({pp(x.delta)})' for x in tag.tail(3).iloc[::-1].itertuples())}.",
               f"Idiomas que mudaram mais de 2 p.p. entre os usuários: <b>{e_lista(mudou['idioma']) if len(mudou) else 'nenhum'}</b>. O chinês simplificado oscila muito de um mês para o outro; leia como indício."]
    return dict(id='p5', curto='Tendência', titulo='Quem cresce e quem cai',
                pergunta='Quais gêneros e tags ganharam ou perderam participação, e o peso de cada idioma mudou entre 2022 e hoje?',
                headline=headline, stats=stats, leitura=leitura, figs=[f1, f2, f3], tabelas=[ts],
                nota=f'Participação = % dos lançamentos do período (jogos com gênero ou tag). Tags com pelo menos {MIN_TAG} lançamentos em cada período. Jogos de 2026 ficam fora. Survey de outubro/2026 conferido na página oficial da Valve.',
                dados=dict(gen=gen, tag=tag, surv=surv))

P5_ = pergunta5()
print('Pergunta 5 ok')

Pergunta 5 ok


## 9. Pergunta 6 — Japonês x Ocidente (métrica 3 com grupos exclusivos)

In [9]:
def pergunta6():
    grupos = {'Japonês (com ou sem outros)': g['lang_japanese'],
              'Inglês sem japonês': g['lang_english'] & ~g['lang_japanese'],
              'Alemão sem japonês': g['lang_german'] & ~g['lang_japanese']}
    nomes = list(grupos)
    partes, tam = [], {}
    for nome, mask in grupos.items():
        sub = g[mask]
        tam[nome] = len(sub)
        total = sub.loc[sub['genres_str'].notna(), 'estimated_owners_mid'].sum()
        d = explodir(sub, 'genres_str')
        r = d.groupby('item')['estimated_owners_mid'].sum().reset_index()
        r['part'] = r['estimated_owners_mid'] / total * 100
        r['grupo'] = nome
        partes.append(r)
    p = pd.concat(partes).pivot(index='item', columns='grupo', values='part')[nomes].fillna(0)
    jp, en, de = nomes
    p['dif_en'] = p[jp] - p[en]; p['dif_de'] = p[jp] - p[de]
    p['destaque'] = (p['dif_en'].abs() > 5) | (p['dif_de'].abs() > 5)
    p = p.sort_values(jp, ascending=False)

    f1 = go.Figure()
    for nome, cor in zip(nomes, [LARANJA, AZUL, CINZA]):
        f1.add_bar(name=f'{nome} ({n0(tam[nome])} jogos)', x=p.index, y=p[nome], marker_color=cor)
    f1.update_layout(barmode='group', yaxis_title='Participação nos donos (%)')
    estilo(f1, 'Perfil de gêneros: japonês x inglês sem japonês x alemão sem japonês', 450)

    q = p.sort_values('dif_en')
    f2 = make_subplots(rows=1, cols=2, shared_yaxes=True, subplot_titles=('Japonês menos inglês sem japonês', 'Japonês menos alemão sem japonês'))
    f2.add_bar(y=q.index, x=q['dif_en'], orientation='h', marker_color=[VERDE if v >= 0 else VERMELHO for v in q['dif_en']], row=1, col=1)
    f2.add_bar(y=q.index, x=q['dif_de'], orientation='h', marker_color=[VERDE if v >= 0 else VERMELHO for v in q['dif_de']], row=1, col=2)
    for c in (1, 2):
        f2.add_vline(x=5, line_dash='dot', line_color=MUTED, row=1, col=c)
        f2.add_vline(x=-5, line_dash='dot', line_color=MUTED, row=1, col=c)
        f2.update_xaxes(title_text='pontos percentuais', row=1, col=c)
    estilo(f2, 'Diferença de participação em pontos percentuais (linhas pontilhadas: ±5 p.p.)', 420, False)

    t = p.reset_index().rename(columns={'item': 'Gênero'})
    for c in [jp, en, de]: t[c] = t[c].map(pct)
    for c in ['dif_en', 'dif_de']: t[c] = t[c].map(lambda v: pp(v, 1))
    t['destaque'] = t['destaque'].map({True: 'mais de 5 p.p.', False: '–'})
    t.columns = ['Gênero', 'Japonês', 'Inglês s/ jap.', 'Alemão s/ jap.', 'Dif. vs inglês', 'Dif. vs alemão', 'Destaque']

    dest = p[p['destaque']]
    maior = p['dif_en'].abs().idxmax()
    headline = f"O perfil do mercado japonês difere do ocidental em {len(dest)} de {len(p)} gêneros (mais de 5 p.p. em algum dos dois comparativos)."
    stats = [(n0(tam[jp]), 'jogos com suporte a japonês'),
             (pp(p.loc[maior, 'dif_en'], 1), f'{maior}: maior diferença do japonês para o inglês sem japonês'),
             (f'{len(dest)} de {len(p)}', 'gêneros com diferença acima de 5 p.p.')]
    leitura = [f"Tamanho dos grupos: " + ' · '.join(f'{k}: {n0(v)} jogos' for k, v in tam.items()) + '.'] + \
              [f"<b>{i}</b>: {pp(r.dif_en, 1)} vs inglês, {pp(r.dif_de, 1)} vs alemão" for i, r in dest.iterrows()]
    return dict(id='p6', curto='Japão', titulo='Japão x Ocidente',
                pergunta='O perfil de gêneros dos jogos com japonês difere do dos jogos com inglês (sem japonês) e alemão (sem japonês)?',
                headline=headline, stats=stats, leitura=leitura, figs=[f1, f2], tabelas=[t],
                nota='Os grupos foram separados para não se sobreporem ao japonês; inglês sem japonês e alemão sem japonês ainda podem ter jogos em comum (os que têm os dois idiomas). Participação = donos do gênero ÷ donos do grupo.',
                dados=dict(p=p, tam=tam, dest=dest))

P6_ = pergunta6()
print('Pergunta 6 ok')

Pergunta 6 ok


## 10. Solução para a publisher
Esta célula transforma os resultados das 6 perguntas em recomendações. Todo número do texto vem dos dados calculados acima.

In [10]:
PERGUNTAS = [P1_, P2_, P3_, P4_, P5_, P6_]

def montar_solucao():
    r4 = P4_['dados']['r']; r3 = P3_['dados']['todas']; top5 = P3_['dados']['top5']
    gen = P5_['dados']['gen']; tag = P5_['dados']['tag']; surv = P5_['dados']['surv']
    rho = P2_['dados']['rho']

    investir, evitar = [], []

    multi = r4[(r4['grupo'] != 'Singleplayer') & (r4['taxa_hits'] > TAXA_BASE)]
    if len(multi):
        sp = r4.loc[r4['grupo'] == 'Singleplayer', 'taxa_hits']
        ev = ' · '.join(f'{x.grupo} {pct(x.taxa_hits)}' for x in multi.itertuples())
        ev += f' contra {pct(sp.iloc[0])} do Singleplayer e {pct(TAXA_BASE)} da base.' if len(sp) else f' contra {pct(TAXA_BASE)} da base.'
        investir.append(('Desenvolver com modo multijogador ou cooperativo', ev))
    investir.append(('Escolher gênero e preço pela taxa de hits, não pelo tamanho do gênero',
                     'Melhores combinações: ' + '; '.join(f'{x.item} · {x.price_band} ({pct(x.taxa_hits)})' for x in top5.head(3).itertuples()) + f'. Base: {pct(TAXA_BASE)}.'))
    investir.append(('Mirar gêneros que ganham espaço nos lançamentos',
                     'Gêneros: ' + ', '.join(f'{x.nome} ({pp(x.delta)})' for x in gen.head(3).itertuples()) +
                     '. Tags: ' + ', '.join(f'{x.nome} ({pp(x.delta)})' for x in tag.head(3).itertuples()) + '.'))

    evitar.append(('Entrar em gêneros e tags em queda',
                   'Gêneros: ' + ', '.join(f'{x.nome} ({pp(x.delta)})' for x in gen.tail(3).iloc[::-1].itertuples()) +
                   '. Tags: ' + ', '.join(f'{x.nome} ({pp(x.delta)})' for x in tag.tail(3).iloc[::-1].itertuples()) + '.'))
    evitar.append(('Contar só com a nota dos jogadores para gerar público',
                   f'Correlação entre % positivas e donos: ρ = {n2(rho)} ({forca(rho)}). A nota ajuda a reputação, mas não substitui divulgação e alcance.'))
    pior = r3.sort_values('taxa_hits').head(3)
    evitar.append(('Combinações de gênero e preço com baixa taxa de hits',
                   'Menores taxas: ' + '; '.join(f'{x.item} · {x.price_band} ({pct(x.taxa_hits)})' for x in pior.itertuples()) + '.'))

    # tabela por mercado
    lid = P1_['dados']['lideres']; of = P1_['dados']['oferta']
    linhas = ''
    for _, row in surv.iterrows():
        m = row['idioma']
        linhas += (f"<tr><td>{m}</td><td>{', '.join(lid[m]['genero'])}</td>"
                   f"<td>{pct(row['abr22'])} → {pct(row['out26'])} ({pp(row['var'])})</td><td>{pct(of[m])}</td></tr>")
    dest = P6_['dados']['dest']
    nota_jp = (f"No japonês, o perfil de gêneros difere do ocidental em {e_lista(dest.index)}; vale tratar o lançamento no Japão com estratégia própria."
               if len(dest) else 'O perfil de gêneros do japonês não difere do ocidental acima de 5 p.p.')

    def coluna(titulo, itens, cls):
        li = ''.join(f'<li><b>{a}</b><span>{b}</span></li>' for a, b in itens)
        return f'<div class="col {cls}"><h3>{titulo}</h3><ol>{li}</ol></div>'

    return (f'<div class="duas">{coluna("Onde investir", investir, "sim")}{coluna("O que evitar", evitar, "nao")}</div>'
            '<h3 class="sub">Leitura por mercado de idioma</h3>'
            '<table class="tabela"><thead><tr><th>Mercado</th><th>Gêneros líderes (donos)</th><th>Usuários da Steam, abr/2022 → out/2026</th><th>Jogos com o idioma</th></tr></thead>'
            f'<tbody>{linhas}</tbody></table><p class="nota">{nota_jp}</p>')

SOLUCAO_HTML = montar_solucao()
print('Solução montada')

Solução montada


## 11. Montar a apresentação
Gera a página com abas (Contexto, 6 perguntas e Solução). Use as setas do teclado ou os botões para navegar. O arquivo `dashboard_steam.html` é salvo e exibido abaixo.

In [11]:
CSS = """
<style>
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600&family=Source+Serif+4:opsz,wght@8..60,500;8..60,600&display=swap');
:root{--bg:#f6f4ef;--ink:#1f2428;--mut:#6b6f73;--line:#e4e0d6;--ac:#c8553d;--az:#2f5d7c}
*{box-sizing:border-box}
body{margin:0;background:var(--bg);color:var(--ink);font-family:Inter,'Segoe UI',Arial,sans-serif;font-size:15px;line-height:1.55}
nav{position:sticky;top:0;z-index:10;background:var(--bg);border-bottom:1px solid var(--line);display:flex;align-items:center;gap:4px;padding:0 28px;overflow-x:auto}
nav .marca{font-family:'Source Serif 4',Georgia,serif;font-weight:600;font-size:16px;margin-right:22px;white-space:nowrap}
nav button{background:none;border:0;border-bottom:2px solid transparent;padding:15px 10px;font:500 13px Inter,sans-serif;color:var(--mut);cursor:pointer;white-space:nowrap}
nav button:hover{color:var(--ink)}
nav button.on{color:var(--ink);border-bottom-color:var(--ac)}
.slide{display:none;max-width:1180px;margin:0 auto;padding:40px 28px 50px}
.slide.on{display:block}
.eyebrow{font-size:12px;letter-spacing:.08em;text-transform:uppercase;color:var(--ac);font-weight:600;margin:0 0 10px}
.perg{color:var(--mut);margin:0 0 14px;max-width:820px}
h1,h2{font-family:'Source Serif 4',Georgia,serif;font-weight:600;letter-spacing:-.01em;margin:0}
h1{font-size:44px;line-height:1.1;max-width:880px}
h2{font-size:32px;line-height:1.2;max-width:900px}
.lead{font-size:18px;color:var(--mut);max-width:760px;margin:18px 0 0}
.stats{display:flex;flex-wrap:wrap;margin:30px 0 34px;border-top:1px solid var(--line);border-bottom:1px solid var(--line)}
.stat{flex:1;min-width:210px;padding:18px 24px 18px 0;margin-right:24px}
.stat+.stat{border-left:1px solid var(--line);padding-left:24px}
.stat .n{font-family:'Source Serif 4',Georgia,serif;font-size:38px;font-weight:600;line-height:1.1}
.stat .l{font-size:13px;color:var(--mut);margin-top:6px;max-width:260px}
.principal{display:grid;grid-template-columns:minmax(0,1.7fr) minmax(0,1fr);gap:40px;align-items:start}
.cap{font-size:12px;letter-spacing:.04em;text-transform:uppercase;color:var(--mut);font-weight:600;margin:0 0 4px}
.leitura h3,.sub{font-size:12px;letter-spacing:.08em;text-transform:uppercase;color:var(--mut);font-weight:600;margin:0 0 10px}
.sub{margin-top:36px}
.pontos{list-style:none;margin:0;padding:0}
.pontos li{padding:10px 0;border-bottom:1px solid var(--line);font-size:14px}
.pontos li:first-child{padding-top:0}
.sec{display:grid;grid-template-columns:repeat(auto-fit,minmax(430px,1fr));gap:34px;margin-top:34px;padding-top:26px;border-top:1px solid var(--line)}
details{margin-top:26px;border-top:1px solid var(--line);padding-top:14px}
summary{cursor:pointer;font-size:13px;font-weight:600;color:var(--az)}
.tabela{border-collapse:collapse;width:100%;font-size:13px;margin:12px 0}
.tabela th{text-align:left;font-weight:600;color:var(--mut);font-size:12px;border-bottom:1px solid var(--ink);padding:8px 10px 8px 0}
.tabela td{padding:8px 10px 8px 0;border-bottom:1px solid var(--line);vertical-align:top}
.nota{font-size:12px;color:var(--mut);max-width:900px;margin:14px 0 0}
.rodape{display:flex;justify-content:space-between;margin-top:36px;padding-top:18px;border-top:1px solid var(--line)}
.rodape button{background:none;border:1px solid var(--line);padding:9px 16px;border-radius:4px;font:500 13px Inter,sans-serif;cursor:pointer;color:var(--ink)}
.rodape button:hover{border-color:var(--ink)}
.lista-perg{margin:34px 0 0;padding:0;list-style:none;max-width:900px}
.lista-perg li{display:flex;gap:18px;padding:13px 0;border-bottom:1px solid var(--line);cursor:pointer}
.lista-perg li:hover b{color:var(--ac)}
.lista-perg .num{font-family:'Source Serif 4',Georgia,serif;color:var(--ac);font-weight:600;min-width:22px}
.lista-perg span{color:var(--mut);font-size:14px}
.duas{display:grid;grid-template-columns:repeat(auto-fit,minmax(420px,1fr));gap:48px;margin-top:30px}
.col h3{font-family:'Source Serif 4',Georgia,serif;font-size:24px;margin:0 0 6px;padding-bottom:10px;border-bottom:2px solid var(--ink)}
.col.sim h3{border-bottom-color:var(--az)} .col.nao h3{border-bottom-color:var(--ac)}
.col ol{margin:0;padding:0;list-style:none;counter-reset:i}
.col li{counter-increment:i;padding:16px 0 16px 38px;border-bottom:1px solid var(--line);position:relative}
.col li:before{content:counter(i);position:absolute;left:0;top:15px;font-family:'Source Serif 4',Georgia,serif;font-size:22px;font-weight:600;color:var(--mut)}
.col li b{display:block;font-size:15.5px;margin-bottom:3px}
.col li span{font-size:13.5px;color:var(--mut)}
@media(max-width:860px){.principal{grid-template-columns:1fr}.sec,.duas{grid-template-columns:1fr}h1{font-size:34px}h2{font-size:26px}}
</style>
"""

JS = """
<script>
var ORDEM = %s, ATUAL = 0;
function ir(i){
  if(i<0||i>=ORDEM.length) return; ATUAL=i;
  document.querySelectorAll('.slide').forEach(function(e){e.classList.remove('on')});
  document.querySelectorAll('nav button').forEach(function(e){e.classList.remove('on')});
  document.getElementById(ORDEM[i]).classList.add('on');
  document.getElementById('b_'+ORDEM[i]).classList.add('on');
  setTimeout(function(){document.querySelectorAll('#'+ORDEM[i]+' .plotly-graph-div').forEach(function(d){Plotly.Plots.resize(d)})},60);
  window.scrollTo(0,0);
}
document.addEventListener('keydown',function(e){if(e.key==='ArrowRight')ir(ATUAL+1);if(e.key==='ArrowLeft')ir(ATUAL-1)});
</script>
"""

def rodape(i, ordem, nomes):
    ant = f'<button onclick="ir({i-1})">← {nomes[i-1]}</button>' if i > 0 else '<span></span>'
    prox = f'<button onclick="ir({i+1})">{nomes[i+1]} →</button>' if i < len(ordem) - 1 else '<span></span>'
    return f'<div class="rodape">{ant}{prox}</div>'

def montar_dashboard():
    ordem = ['ctx'] + [p['id'] for p in PERGUNTAS] + ['sol']
    nomes = ['Contexto'] + [f"{n}. {p['curto']}" for n, p in enumerate(PERGUNTAS, 1)] + ['Solução']
    nav = '<nav><span class="marca">Mercado de games na Steam</span>' + ''.join(
        f'<button id="b_{o}" onclick="ir({k})"{" class=on" if k == 0 else ""}>{n}</button>' for k, (o, n) in enumerate(zip(ordem, nomes))) + '</nav>'

    # ---- Slide de contexto
    stats = [(n0(len(g)), 'jogos na base'),
             (pct(TAXA_BASE), f'taxa geral de hits (mais de 100 mil donos): {n0(g["is_hit"].sum())} jogos'),
             (n0(g['estimated_owners_mid'].median()), 'mediana de donos estimados por jogo'),
             (pct(g['price_band'].eq('gratuito').mean() * 100), 'dos jogos são gratuitos')]
    st = ''.join(f'<div class="stat"><div class="n">{v}</div><div class="l">{l}</div></div>' for v, l in stats)
    lista = ''.join(f'<li onclick="ir({k})"><span class="num">{k}</span><div><b>{p["titulo"]}</b><br><span>{p["pergunta"]}</span></div></li>'
                    for k, p in enumerate(PERGUNTAS, 1))
    ctx = (f'<section class="slide on" id="ctx"><p class="eyebrow">Projeto final · Análise de Dados e BI</p>'
           f'<h1>Onde investir no próximo lançamento de games</h1>'
           f'<p class="lead">O que faz um jogo ter público na Steam: gênero, preço, nota, modo de jogo e mercado de idioma. '
           f'Público é medido em donos estimados (Steam Spy), não em vendas.</p>'
           f'<div class="stats">{st}</div><p class="cap">As seis perguntas</p><ul class="lista-perg">{lista}</ul>'
           f'<p class="nota">Bruno Sena, Matheus Gabriel e Pedro Vitor · Fontes: Steam Games Dataset (Kaggle) e Steam Hardware &amp; Software Survey (Valve)</p>'
           f'{rodape(0, ordem, nomes)}</section>')

    primeiro = True
    def fig_html(f):
        nonlocal primeiro
        h = f.to_html(full_html=False, include_plotlyjs='cdn' if primeiro else False,
                      config={'displayModeBar': False, 'responsive': True})
        primeiro = False
        return f'<p class="cap">{f.layout.meta}</p>{h}'

    slides = ''
    for k, p in enumerate(PERGUNTAS, 1):
        st = ''.join(f'<div class="stat"><div class="n">{v}</div><div class="l">{l}</div></div>' for v, l in p['stats'])
        pontos = ''.join(f'<li>{x}</li>' for x in p['leitura'])
        sec = ''.join(f'<div>{fig_html(f)}</div>' for f in p['figs'][1:])
        tabs = ''.join(tabela_html(t) for t in p['tabelas'])
        slides += (f'<section class="slide" id="{p["id"]}"><p class="eyebrow">Pergunta {k} de {len(PERGUNTAS)} · {p["titulo"]}</p>'
                   f'<p class="perg">{p["pergunta"]}</p><h2>{p["headline"]}</h2><div class="stats">{st}</div>'
                   f'<div class="principal"><div>{fig_html(p["figs"][0])}</div><div class="leitura"><h3>Como ler</h3><ul class="pontos">{pontos}</ul></div></div>'
                   f'<div class="sec">{sec}</div><details><summary>Ver tabelas de dados</summary>{tabs}</details>'
                   f'<p class="nota">{p["nota"]}</p>{rodape(k, ordem, nomes)}</section>')

    sol = (f'<section class="slide" id="sol"><p class="eyebrow">Solução</p>'
           f'<h2>O que a publisher deve fazer no próximo lançamento</h2>'
           f'<p class="lead">Recomendações tiradas das seis análises. Público = donos estimados; os resultados indicam associação, não causa.</p>'
           f'{SOLUCAO_HTML}{rodape(len(ordem) - 1, ordem, nomes)}</section>')

    js = JS % str(ordem)
    return f'<html><head><meta charset="utf-8">{CSS}</head><body>{nav}{ctx}{slides}{sol}{js}</body></html>'

html = montar_dashboard()
with open('dashboard_steam.html', 'w', encoding='utf-8') as f:
    f.write(html)
display(HTML(html))

Mercado,#,Gênero,Participação (% donos),Mediana donos,Mediana donos (>20 mil),Taxa de hits,Jogos
Inglês,1,Action,"64,6%",10.000,75.000,"9,2%",46.283
Inglês,2,Indie,"44,7%",10.000,75.000,"6,5%",79.500
Inglês,3,Adventure,"42,6%",10.000,75.000,"8,1%",43.828
Alemão,1,Action,"69,0%",10.000,150.000,"20,3%",12.421
Alemão,2,Adventure,"43,2%",10.000,75.000,"17,5%",12.229
Alemão,3,Indie,"35,5%",10.000,75.000,"12,6%",20.554
Japonês,1,Action,"69,3%",10.000,150.000,"15,0%",12.197
Japonês,2,Adventure,"45,1%",10.000,75.000,"13,8%",11.972
Japonês,3,Indie,"38,8%",10.000,75.000,"10,1%",20.775
Chinês simplificado,1,Action,"67,8%",10.000,75.000,"14,8%",12.761


## 12. Baixar a apresentação (opcional)
O arquivo `dashboard_steam.html` abre em qualquer navegador (precisa de internet para carregar o Plotly e as fontes). Para apresentar: abra o arquivo, tecle F11 e use as setas ← →.

In [12]:
from google.colab import files
files.download('dashboard_steam.html')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>